# 12 Bark Detector

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from woofdata.loader import load_sample
from woofdata.path import data_path, load_data_dir_from_env_file
from woofdata.tasks import read_manifest, task_rows
from woofdecoder.branches.bark_detector import (
    FRAME_S,
    PRETRAINED_SED_DIR,
    detect_sed_probs,
    frames_to_events,
    load_bark_detector,
)
from woofdecoder.preprocess.media import load_audio_16k
from woofdecoder.sed_classes import SED_CLASSES
from wooftrainer.eval.metrics import event_scores, frame_scores

TASK = "bark_detector"
CACHE_PATH = data_path("cache/bark_detector/playpant_val_zero_shot.npz")

## Pretrained SED Check 

In [ ]:
load_data_dir_from_env_file()

print(f"{PRETRAINED_SED_DIR}")
print(f"Exists: {PRETRAINED_SED_DIR.is_dir()}")

## Dataset Check

In [ ]:
manifest = read_manifest()
for source in ["playpant", "audioset_strong_eval", "esc50"]:
    counts = manifest.loc[manifest["source"] == source, "split"].value_counts()
    print(f"{source:<22} {dict(counts)}")

audioset_rows = manifest[manifest["source"] == "audioset_strong_eval"]
print(audioset_rows["original_label"].value_counts().head(15).to_string())

## Load Bark Detector

### Basic Check on Test Video

In [ ]:
detector = load_bark_detector()

audio = load_audio_16k(data_path("test/media_test.mp4"))
start = time.perf_counter()
sed_probs = detect_sed_probs(detector, audio)
seconds_taken = time.perf_counter() - start
print(f"Output shape: {sed_probs.shape} (frames x {SED_CLASSES})")
print(f"{len(audio) / 16_000:.1f} s of audio took {seconds_taken:.2f} s")

### Show Events above 0.5


In [ ]:
dog_prob = sed_probs.max(axis=1)
for start_s, end_s in frames_to_events(dog_prob >= 0.5):
    print(f"{start_s:6.2f} - {end_s:6.2f} s")

times = np.arange(len(sed_probs)) * FRAME_S
plt.figure(figsize=(14,3))
for column, sed_class in enumerate(SED_CLASSES):
    plt.plot(times, sed_probs[:, column], label=sed_class, linewidth=1)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("time (s)")
plt.ylabel("probability")
plt.title("Bark detector on test video")
plt.legend(ncol=6)
plt.show()

## Baseline on playpant

In [ ]:
val_rows = task_rows(TASK, "val", sources = ["playpant"])
print(f"{len(val_rows)} windows of 10 s\n")

all_probs, all_truth = [], []
for number, (_,row) in enumerate(val_rows.iterrows(), start =1):
    sample = load_sample(TASK, row)
    sed_probs = detect_sed_probs(detector, sample.audio)[: len(sample.targets)]
    all_probs.append(sed_probs)
    all_truth.append(sample.targets)
    if number % 100 == 0 or number == len(val_rows):
        print(f"{number}/{len(val_rows)} done")
        
val_probs = np.stack(all_probs)
val_targets = np.stack(all_truth)

CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez(CACHE_PATH, probs=val_probs, targets=val_targets, clip_ids=val_rows["clip_id"].to_numpy())
print(f"\nSaved to {CACHE_PATH}")

true_dog = val_targets.any(axis=2)
print(f"Frames with a dog sound: {true_dog.mean():.1%}")
for column, sed_class in enumerate(SED_CLASSES):
    print(f"{sed_class:<12} {val_targets[:, :, column].mean():.2%} of frames")
    
dog_prob = val_probs.max(axis=2)  # shape (num_windows, 250)
print(f"\n{'threshold':>9} | {'frame P':>7} {'frame R':>7} {'frame F1':>8} | "
      f"{'event P':>7} {'event R':>7} {'event F1':>8} {'#pred':>6}")

for threshold in np.arange(0.1, 1.0, 0.1):
    predicted = dog_prob >= threshold
    frames = frame_scores(true_dog, predicted)
    events = event_scores(true_dog, predicted)
    print(f"{threshold:>9.1f} | {frames['precision']:>7.2f} {frames['recall']:>7.2f} "
          f"{frames['f1']:>8.2f} | {events['precision']:>7.2f} {events['recall']:>7.2f} "
          f"{events['f1']:>8.2f} {events['num_predicted']:>6}")
    
print(f"True events in val: {events['num_true']}\n")

for column, sed_class in enumerate(SED_CLASSES):
    truth = val_targets[:, :, column] > 0
    if not truth.any():
        print(f"{sed_class:<12} no true frames in val, skipped")
        continue
    scores = frame_scores(truth, val_probs[:, :, column] >= 0.5)
    print(f"{sed_class:<12} frame P {scores['precision']:.2f}  R {scores['recall']:.2f}  "
          f"F1 {scores['f1']:.2f}")

### Analysis

As the threshold increases, the precision of the model's prediction also increases (meaning that it rarely raises false alarms, so we can trust its prediction). However, as the threshold increases, the recall of the model's prediction significantly decreases (meaning that many SED classes are missed). 

This is expected because most pants in whine are quiet.

The pretrained SED is good at barks, but almost never fires at pant. Because the playpant dataset contains almost all pant data, the numbers show extremely poor performance for the model's prediction. 